# 1 环境依赖

Use the environment of Nimbus, follow the installation steps in github https://github.com/angelolab/Nimbus

Clone the repository

`git clone https://github.com/angelolab/Nimbus.git`

Make a conda environment for Nimbus and activate it

`conda create -n Nimbus python==3.10`

`conda activate Nimbus`

Install CUDA libraries if you have a NVIDIA GPU available

`conda install -c conda-forge cudatoolkit=11.2 cudnn=8.1.0`

Install the package and all depedencies in the conda environment

`python -m pip install -e Nimbus`

# 2 数据目录结构

在每个slides的独立的文件夹下建立一个`segmentation/Nimbus_input`用来储存去重后的mask图像

```
Experiment Folder
|
|---Slides1.ome.tif
|---Slides2.ome.tif
|---Slides3.ome.tif
...
|---Slides1
|   |---Tiles
|       |---FOV0.tif
|       |---FOV1.tif
|       ...
|   |---image_data
|       |---FOV0
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|       |---FOV1
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|   |---segementation
|       |---cellpose_input
|           |---FOV0.tif
|           |---FOV1.tif
|           ...
|       |---cellpose_output
|           |---FOV0.tif
|           |---FOV1.tif
|           ...
|       |---Nimbus_input
|           |---FOV0.tif
|           |---FOV1.tif
|           ...
...
```

# 3 处理逻辑

## Mask Edge Removement

清除所有与图像边缘相接的mask

## Stitch Mask

相邻FOV的Mask

# 4 原始代码

## Mask Edge Removement
/home/jqyang/Nimbus-Inference/templates/Mask_Edge_RM_Cellpose.ipynb


In [1]:
import os
import glob
import numpy as np
from skimage.segmentation import clear_border
from skimage.io import imread, imsave

# --- 1. 全局配置 ---

# 数据集的根目录 (list.txt 中的文件夹应该在这个目录下)
DATA_ROOT = "../data"

# 包含样本文件夹名称列表的 txt 文件路径
# list.txt 内容示例:
# B221D-TI-N-cellpose
# B222D-TI-N-cellpose
LIST_FILE_PATH = "list.txt"

# 输入图片的子目录名 (读取 mask 的位置)
INPUT_SUBFOLDER_NAME = "segmentation/cellpose_output"

# 输出图片的子目录名 (保存处理后 mask 的位置)
OUTPUT_SUBFOLDER_NAME = "segmentation/deepcell_output"

# --- 结束配置 ---

def process_single_sample(sample_name):
    """
    处理单个样本文件夹：
    1. 读取输入目录中的 mask
    2. 移除边缘细胞
    3. 保存到新的输出目录 (deepcell_output)
    """
    
    # 构建完整的输入和输出路径
    input_dir = os.path.join(DATA_ROOT, sample_name, INPUT_SUBFOLDER_NAME)
    output_dir = os.path.join(DATA_ROOT, sample_name, OUTPUT_SUBFOLDER_NAME)

    print(f"\n{'='*50}")
    print(f"正在处理样本: {sample_name}")
    print(f"  📂 输入目录: {input_dir}")
    print(f"  📂 输出目录: {output_dir}")

    # 检查输入目录是否存在
    if not os.path.exists(input_dir):
        print(f"  ❌ 跳过：找不到输入目录 -> {input_dir}")
        return

    # 创建输出目录 (如果不存在)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"  ✅ 已创建输出目录: {output_dir}")

    # 查找符合条件的文件
    pattern_cell = os.path.join(input_dir, "fov*_whole_cell.tiff")
    cell_files = glob.glob(pattern_cell)

    if not cell_files:
        print(f"  ⚠️ 警告：输入目录中没有找到 'fov*_whole_cell.tiff' 文件。")
        return

    print(f"  📊 找到 {len(cell_files)} 个文件，开始处理...")

    # 循环处理当前样本下的所有文件
    success_count = 0
    for i, file_path in enumerate(cell_files):
        filename = os.path.basename(file_path)
        
        try:
            # (A) 读取
            mask_image = imread(file_path)
            
            # (B) 处理 (移除边缘)
            cleared_mask = clear_border(mask_image, buffer_size=2)
            
            # (C) 保存到新路径
            save_path = os.path.join(output_dir, filename)
            imsave(save_path, cleared_mask, check_contrast=False)
            
            success_count += 1
            # 可选：打印进度 (文件太多时可以注释掉下面这行)
            # print(f"    Processing {filename} -> Done.")

        except Exception as e:
            print(f"    ❌ 处理文件 {filename} 失败: {e}")

    print(f"  ✅ 样本 {sample_name} 处理完成: 成功保存 {success_count}/{len(cell_files)} 张图像。")


def main():
    # 1. 检查 list.txt 是否存在
    if not os.path.exists(LIST_FILE_PATH):
        print(f"❌ 错误：找不到列表文件: {LIST_FILE_PATH}")
        print("请创建一个包含样本文件夹名称的 list.txt 文件。")
        return

    # 2. 读取样本列表
    print(f"📖 正在读取样本列表: {LIST_FILE_PATH} ...")
    with open(LIST_FILE_PATH, 'r') as f:
        # 读取每一行，去除空白字符，忽略空行
        samples = [line.strip() for line in f if line.strip()]

    print(f"📋 待处理样本总数: {len(samples)}")

    # 3. 遍历列表进行处理
    for sample in samples:
        process_single_sample(sample)

    print(f"\n{'='*50}")
    print("🎉 所有样本批量处理结束！")

if __name__ == "__main__":
    main()

📖 正在读取样本列表: list.txt ...
📋 待处理样本总数: 10

正在处理样本: A099A-CXCR4-cellpose
  📂 输入目录: ../data/A099A-CXCR4-cellpose/segmentation/cellpose_output
  📂 输出目录: ../data/A099A-CXCR4-cellpose/segmentation/deepcell_output
  ✅ 已创建输出目录: ../data/A099A-CXCR4-cellpose/segmentation/deepcell_output
  📊 找到 71 个文件，开始处理...
  ✅ 样本 A099A-CXCR4-cellpose 处理完成: 成功保存 71/71 张图像。

正在处理样本: B274B-CXCR4-cellpose
  📂 输入目录: ../data/B274B-CXCR4-cellpose/segmentation/cellpose_output
  📂 输出目录: ../data/B274B-CXCR4-cellpose/segmentation/deepcell_output
  ✅ 已创建输出目录: ../data/B274B-CXCR4-cellpose/segmentation/deepcell_output
  📊 找到 54 个文件，开始处理...
  ✅ 样本 B274B-CXCR4-cellpose 处理完成: 成功保存 54/54 张图像。

正在处理样本: B319D-CXCR4-cellpose
  📂 输入目录: ../data/B319D-CXCR4-cellpose/segmentation/cellpose_output
  📂 输出目录: ../data/B319D-CXCR4-cellpose/segmentation/deepcell_output
  ✅ 已创建输出目录: ../data/B319D-CXCR4-cellpose/segmentation/deepcell_output
  📊 找到 63 个文件，开始处理...
  ✅ 样本 B319D-CXCR4-cellpose 处理完成: 成功保存 63/63 张图像。

正在处理样本: B512A-CXCR4-1-cellpose
  

## Stitch Mask

/home/jqyang/Nimbus-Inference/templates/stitch_masks-Copy4_batch.ipynb

In [1]:
import pandas as pd
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm # 如果在 Notebook 中运行，可以改为 from tqdm.notebook import tqdm
import sys
import warnings
import os
import shutil
from skimage.measure import regionprops

# 忽略 tifffile 关于非标准 TIFF 标签的常见警告
warnings.filterwarnings('ignore', message='.*TiffPage.*')

# =============================================================================
# --- 1. 配置参数 ---
# =============================================================================
LIST_FILE = "list.txt"          # 存放样本文件夹名称的文本文件
DATA_ROOT = Path("../data/")    # 数据根目录
PATTERNS_TO_PROCESS = ["{FOV_Name}_whole_cell.tiff"] # 要处理的文件模式
MIN_CELL_SIZE_FILTER = 0        # 最小细胞尺寸过滤
OVERLAP_THRESHOLD = 0.1         # 去重重叠阈值 (10%)

# =============================================================================
# --- 2. 核心逻辑函数 ---
# =============================================================================

def build_virtual_global_mask(csv_file, tile_dir, tiff_pattern):
    """在内存中构建虚拟全局掩码并进行 V3 混合阈值去重"""
    try:
        df = pd.read_csv(csv_file)
    except FileNotFoundError:
        print(f"错误: 坐标文件 {csv_file} 未找到。")
        return None, None, None

    tile_info = []
    for _, row in df.iterrows():
        fov_name = row['FOV_Name']
        file_name = tiff_pattern.format(FOV_Name=fov_name)
        file_path = tile_dir / file_name
        tile_info.append({
            "fov_name": fov_name, "path": file_path,
            "x": int(row['X']), "y": int(row['Y']),
            "w": int(row['W']), "h": int(row['H'])
        })

    max_x_coord = (df['X'] + df['W']).max()
    max_y_coord = (df['Y'] + df['H']).max()
    
    try:
        global_mask = np.zeros((max_y_coord, max_x_coord), dtype=np.uint32)
    except MemoryError:
        print(f"内存错误: 无法分配画布内存。")
        return None, None, None

    sorted_tiles = sorted(tile_info, key=lambda t: (t['y'], t['x']))
    global_cell_id_counter = 1 
    global_id_ranges = {}      
    
    for tile in tqdm(sorted_tiles, desc="  拼接 Tiles", leave=False):
        if not tile['path'].exists():
            global_id_ranges[tile['path']] = (0, 0)
            continue
            
        try:
            local_mask = tifffile.imread(tile['path'])
            x, y = tile['x'], tile['y']
            h_local, w_local = local_mask.shape
            canvas_slice = global_mask[y:y+h_local, x:x+w_local]
            
            local_ids = np.unique(local_mask[local_mask > 0])
            if len(local_ids) == 0:
                global_id_ranges[tile['path']] = (0, 0)
                continue 

            remapped_local_mask = np.zeros_like(local_mask, dtype=np.uint32)
            id_start = global_cell_id_counter
            current_global_id = id_start
            
            # V3 混合去重逻辑
            all_areas = np.bincount(local_mask.ravel())
            conflict_pixels_map = (canvas_slice > 0)
            conflict_counts = np.bincount(local_mask[conflict_pixels_map])

            for local_id in local_ids:
                cell_area = all_areas[local_id]
                conflict_pixel_count = conflict_counts[local_id] if local_id < len(conflict_counts) else 0
                conflict_ratio = conflict_pixel_count / cell_area if cell_area > 0 else 0
                
                if conflict_ratio < OVERLAP_THRESHOLD:
                    remapped_local_mask[local_mask == local_id] = current_global_id
                    current_global_id += 1
            
            global_id_ranges[tile['path']] = (id_start, current_global_id)
            global_cell_id_counter = current_global_id
            
            write_mask = (canvas_slice == 0) & (remapped_local_mask > 0)
            canvas_slice[write_mask] = remapped_local_mask[write_mask]

        except Exception as e:
            print(f"警告: 处理 {tile['path'].name} 时出错: {e}")
            global_id_ranges[tile['path']] = (0, 0)

    return global_mask, tile_info, global_id_ranges

def export_deduplicated_fovs(global_mask, tile_info, global_id_ranges, output_dir, tiff_pattern, min_cell_size=0):
    """从全局掩码中切分并导出去重后的 FOV"""
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for tile in tqdm(tile_info, desc="  导出 FOVs", leave=False):
        try:
            x, y, w, h = tile['x'], tile['y'], tile['w'], tile['h']
            dedup_slice = global_mask[y:y+h, x:x+w]
            
            id_start, id_end = global_id_ranges.get(tile['path'], (0, 0))
            
            filtered_slice = np.zeros_like(dedup_slice, dtype=np.uint32)
            if id_start != 0 or id_end != 0:
                valid_mask = (dedup_slice >= id_start) & (dedup_slice < id_end)
                filtered_slice[valid_mask] = dedup_slice[valid_mask]
            
            # 碎片清理
            if min_cell_size > 0 and filtered_slice.max() > 0:
                props = regionprops(filtered_slice)
                small_ids = [p.label for p in props if p.area < min_cell_size]
                if small_ids:
                    filtered_slice[np.isin(filtered_slice, small_ids)] = 0

            output_path = output_dir / tiff_pattern.format(FOV_Name=tile['fov_name'])
            tifffile.imwrite(output_path, filtered_slice, dtype=np.uint32)
        except Exception as e:
            print(f"警告: 导出 {tile['fov_name']} 时出错: {e}")

# =============================================================================
# --- 3. 批量处理主程序 ---
# =============================================================================

def main():
    if not os.path.exists(LIST_FILE):
        print(f"错误: 找不到列表文件 '{LIST_FILE}'，请创建该文件并填入文件夹名称。")
        return

    with open(LIST_FILE, 'r') as f:
        samples = [line.strip() for line in f.readlines() if line.strip()]

    print(f"开始批量处理，共计 {len(samples)} 个样本...")

    for sample_name in samples:
        print(f"\n>>> 正在处理: {sample_name}")
        
        sample_path = DATA_ROOT / sample_name
        csv_file = sample_path / "fov_coordinates.csv"
        tile_dir = sample_path / "segmentation/deepcell_output/"
        output_tmp_dir = sample_path / "segmentation/deepcell_output_tmp/"

        if not csv_file.exists():
            print(f"    跳过: 找不到坐标文件 {csv_file}")
            continue

        all_success = True
        for pattern in PATTERNS_TO_PROCESS:
            # 阶段 1: 内存拼接
            mask, info, id_map = build_virtual_global_mask(csv_file, tile_dir, pattern)
            
            if mask is not None:
                # 阶段 2: 反向切分导出
                export_deduplicated_fovs(mask, info, id_map, output_tmp_dir, pattern, MIN_CELL_SIZE_FILTER)
            else:
                all_success = False
                break

        # 阶段 4: 最终重命名
        if all_success:
            try:
                bak_dir = Path(str(tile_dir) + "_bak")
                final_dir = Path(str(output_tmp_dir).replace("_tmp", ""))
                
                # 如果备份目录已存在，先删除旧备份
                if bak_dir.exists():
                    shutil.rmtree(bak_dir)
                
                os.rename(tile_dir, bak_dir)
                os.rename(output_tmp_dir, final_dir)
                print(f"--- 样本 {sample_name} 处理成功 ---")
            except Exception as e:
                print(f"*** 重命名样本 {sample_name} 失败: {e} ***")
        else:
            print(f"*** 样本 {sample_name} 处理中途失败，保留原始文件 ***")

    print("\n" + "="*50)
    print("所有样本处理完毕！")
    print("="*50)

if __name__ == "__main__":
    main()

开始批量处理，共计 10 个样本...

>>> 正在处理: A099A-CXCR4-cellpose


--- 样本 A099A-CXCR4-cellpose 处理成功 ---

>>> 正在处理: B274B-CXCR4-cellpose


--- 样本 B274B-CXCR4-cellpose 处理成功 ---

>>> 正在处理: B319D-CXCR4-cellpose


--- 样本 B319D-CXCR4-cellpose 处理成功 ---

>>> 正在处理: B512A-CXCR4-1-cellpose


--- 样本 B512A-CXCR4-1-cellpose 处理成功 ---

>>> 正在处理: B512A-CXCR4-2-cellpose


...

--- 样本 B522A-CXCR4-cellpose 处理成功 ---

>>> 正在处理: B531A-CXCR4-cellpose


--- 样本 B531A-CXCR4-cellpose 处理成功 ---

>>> 正在处理: B533A-CXCR4-cellpose


--- 样本 B533A-CXCR4-cellpose 处理成功 ---

>>> 正在处理: B576A-CXCR4-2-cellpose


--- 样本 B576A-CXCR4-2-cellpose 处理成功 ---

>>> 正在处理: B581A-CXCR4-cellpose


--- 样本 B581A-CXCR4-cellpose 处理成功 ---

所有样本处理完毕！
